# Mini Project 2 Part 1

## 1. My assigned projects (from net2prj.csv)

In [2]:
import pandas as pd, time, requests
from tqdm import tqdm

netid = 'smaciasa'
try:
    prj = pd.read_csv('net2prj.csv')
    mine = prj[prj['netID'] == netid].reset_index(drop=True)
except FileNotFoundError:
    mine = pd.DataFrame({
        'WoC': ['prjemian_spec2nexus','caseywdunn_siphonophore_phylogeny_2017','bc_retina',
                'iic-jku_ddsim','spirit-code_spirit','stanfordnlp_glove',
                'neurodatawithoutborders_matnwb','malbecc_lio','giaf_hpipm',
                'antonyuryev_elsevierapi'],
        'GH': ['https://github.com/prjemian/spec2nexus','https://github.com/caseywdunn/siphonophore_phylogeny_2017',
               'https://github.com/bc/retina','https://github.com/iic-jku/ddsim',
               'https://github.com/spirit-code/spirit','https://github.com/stanfordnlp/GloVe',
               'https://github.com/NeurodataWithoutBorders/matnwb','https://github.com/MALBECC/lio',
               'https://github.com/giaf/hpipm','https://github.com/AntonYuryev/ElsevierAPI']})
    mine['netID'] = netid
print(len(mine), 'projects')
mine

10 projects


,WoC,GH,netID
0,prjemian_spec2nexus,https://github.com/prjemian/spec2nexus,smaciasa
1,caseywdunn_siphonophore_phylogeny_2017,https://github.com/caseywdunn/siphonophore_phy...,smaciasa
2,bc_retina,https://github.com/bc/retina,smaciasa
3,iic-jku_ddsim,https://github.com/iic-jku/ddsim,smaciasa
4,spirit-code_spirit,https://github.com/spirit-code/spirit,smaciasa
5,stanfordnlp_glove,https://github.com/stanfordnlp/GloVe,smaciasa
6,neurodatawithoutborders_matnwb,https://github.com/NeurodataWithoutBorders/matnwb,smaciasa
7,malbecc_lio,https://github.com/MALBECC/lio,smaciasa
8,giaf_hpipm,https://github.com/giaf/hpipm,smaciasa
9,antonyuryev_elsevierapi,https://github.com/AntonYuryev/ElsevierAPI,smaciasa


## 2. GitHub stats: stars, forks, last commit date

In [3]:
GH_TOKEN = None
hdr = {'Accept': 'application/vnd.github+json'}
if GH_TOKEN: hdr['Authorization'] = f'Bearer {GH_TOKEN}'

gh_rows = []
for _, row in mine.iterrows():
    owner, repo = row['GH'].rstrip('/').split('/')[-2:]
    base = f'https://api.github.com/repos/{owner}/{repo}'
    r = requests.get(base, headers=hdr).json()
    c = requests.get(base + '/commits', headers=hdr,
                     params={'per_page': 1, 'sha': r.get('default_branch')}).json()
    last = c[0]['commit']['committer']['date'][:10] if isinstance(c, list) and c else None
    gh_rows.append({'project': row['WoC'], 'nstars': r.get('stargazers_count'),
                    'nforks': r.get('forks_count'), 'lastGHCommitDate': last})
    time.sleep(1)
gh = pd.DataFrame(gh_rows)
print(gh.to_markdown(index=False))

| project                                |   nstars |   nforks | lastGHCommitDate   |
|:---------------------------------------|---------:|---------:|:-------------------|
| prjemian_spec2nexus                    |        3 |        7 | 2026-07-20         |
| caseywdunn_siphonophore_phylogeny_2017 |        2 |        1 | 2024-10-15         |
| bc_retina                              |       14 |        3 | 2025-01-20         |
| iic-jku_ddsim                          |      162 |       41 | 2026-09-28         |
| spirit-code_spirit                     |      144 |       60 | 2023-03-17         |
| stanfordnlp_glove                      |     7230 |     1545 | 2025-07-27         |
| neurodatawithoutborders_matnwb         |       62 |       35 | 2026-09-27         |
| malbecc_lio                            |       29 |       18 | 2024-01-22         |
| giaf_hpipm                             |      708 |      144 | 2026-09-18         |
| antonyuryev_elsevierapi                |      nan | 

### GitHub stats

| project                                |   nstars |   nforks | lastGHCommitDate   |
|:---------------------------------------|---------:|---------:|:-------------------|
| prjemian_spec2nexus                    |        3 |        7 | 2026-07-20         |
| caseywdunn_siphonophore_phylogeny_2017 |        2 |        1 | 2024-10-15         |
| bc_retina                              |       14 |        3 | 2025-01-20         |
| iic-jku_ddsim                          |      162 |       41 | 2026-09-28         |
| spirit-code_spirit                     |      144 |       60 | 2023-03-17         |
| stanfordnlp_glove                      |     7230 |     1545 | 2025-07-27         |
| neurodatawithoutborders_matnwb         |       62 |       35 | 2026-09-27         |
| malbecc_lio                            |       29 |       18 | 2024-01-22         |
| giaf_hpipm                             |      708 |      144 | 2026-09-18         |
| antonyuryev_elsevierapi                |      nan |      nan |        (GitHub repo deleted)            |

## 3. WoC: get commit SHA1s for each project (p2c)

In [4]:
from urllib.parse import quote_plus

class WocRemote:
    """Minimal World of Code API client (same endpoints python-woc's WocMapsRemote uses)."""
    def __init__(self, base_url='https://worldofcode.org/api', api_key=None):
        self.base = base_url.rstrip('/')
        self.s = requests.Session()
        if api_key:
            self.s.headers['Authorization'] = f'Bearer {api_key}'

    def _get(self, path):
        for attempt in range(5):
            r = self.s.get(self.base + path, timeout=60)
            if r.status_code == 429:
                wait = int(r.headers.get('Retry-After', 30))
                print(f'rate limited, waiting {wait}s'); time.sleep(wait); continue
            if r.status_code in (400, 404):
                raise KeyError(r.text)
            r.raise_for_status()
            return r.json()
        raise RuntimeError('too many rate-limit retries: ' + path)

    def get_values(self, map_name, key):
        """All values for a key, following pagination (big projects span several pages)."""
        out, cursor = [], 0
        while cursor is not None:
            r = self._get(f'/lookup/map/{map_name}/{quote_plus(key)}?cursor={cursor}')
            out += r['data']
            cursor = r.get('nextCursor')
            if cursor is not None: time.sleep(1)
        return out

    def get_values_many(self, map_name, keys):
        """Batch lookup, max 10 keys per request. Returns (results, errors)."""
        q = '&'.join(f'q={quote_plus(k)}' for k in keys)
        r = self._get(f'/lookup/map/{map_name}?{q}')
        return r.get('data', {}), r.get('errors', {})

woc = WocRemote()

list_df = []
for p in mine['WoC']:
    for attempt in range(3):
        try:
            shas = woc.get_values('p2c', p)
            break
        except Exception as e:
            print('retry', p, e); time.sleep(5)
    else:
        print('FAILED to get commits for', p); continue
    d = pd.DataFrame(sorted(set(shas)), columns=['sha1'])
    d['project'] = p
    list_df.append(d)
    print(f'{p}: {len(d)} commits')
    time.sleep(1)

df = pd.concat(list_df, ignore_index=True)
df.to_csv('df_commits.csv', index=False)
print('total', len(df))

prjemian_spec2nexus: 1407 commits
caseywdunn_siphonophore_phylogeny_2017: 440 commits
bc_retina: 385 commits
iic-jku_ddsim: 3226 commits
spirit-code_spirit: 3003 commits
stanfordnlp_glove: 263 commits
neurodatawithoutborders_matnwb: 3612 commits
malbecc_lio: 3276 commits
giaf_hpipm: 1600 commits
antonyuryev_elsevierapi: 733 commits
total 17945


## 4. WoC: get commit contents in batches of 10 (with waits)

In [5]:
def fetch(shas):
    out, failed = {}, []
    chunks = [shas[i:i+10] for i in range(0, len(shas), 10)]
    for chunk in tqdm(chunks):
        try:
            res, err = woc.get_values_many('commit.tch', chunk)
            if err:
                failed += list(err.keys()) if isinstance(err, dict) else list(err)
            for k, v in res.items():
                out[k] = v[0]
        except Exception as e:
            print('batch error:', e); failed += chunk; time.sleep(5)
        time.sleep(1)
    return out, failed

all_shas = df['sha1'].unique().tolist()
commits, failed = fetch(all_shas)
# retry failures up to 3 times
for _ in range(3):
    if not failed: break
    print('retrying', len(failed), 'failed commits'); time.sleep(10)
    more, failed = fetch(failed)
    commits.update(more)
print(f'got {len(commits)} / {len(all_shas)} commits; still failed: {len(failed)}')

 68%|██████▊   | 1219/1795 [21:16<09:58,  1.04s/it]

rate limited, waiting 60s


100%|██████████| 1795/1795 [32:15<00:00,  1.08s/it]


retrying 42 failed commits


100%|██████████| 5/5 [00:05<00:00,  1.05s/it]


retrying 42 failed commits


100%|██████████| 5/5 [00:05<00:00,  1.05s/it]


retrying 42 failed commits


100%|██████████| 5/5 [00:05<00:00,  1.05s/it]

got 17903 / 17945 commits; still failed: 42


In [6]:

next(iter(commits.items()))

('00265d6e620c2cdf1aa6c01c950378d5efb0a1a0',
 ['bdbc3a1ec14a11679597320f2963677d97529d46',
  ['984bcd35a4cb174dee0a28200682d8953bd25164',
   '62adf2d5191f067225a8f0bee64b35337b0fc4f7'],
  ['Pete R Jemian <prjemian@users.noreply.github.com>', '1561068026', '-0500'],
  ['GitHub <noreply@github.com>', '1561068026', '-0500'],
  'Merge pull request #160 from prjemian/ignore_pydev_config\n\nignore eclipse PyDev config'])

## 5. Build the summary table and save the CSV

In [7]:
rows = []
for _, r in df.iterrows():
    c = commits.get(r['sha1'])
    if c is None: continue
    rows.append({'project_wocid': r['project'],
                 'commit_sha1': r['sha1'],
                 'author': c[2][0],
                 'time': int(c[2][1]),
                 'commit message': c[4]})
summary = pd.DataFrame(rows).sort_values(['project_wocid', 'time'])

# commit messages contain newlines/semicolons -> flatten newlines; pandas quotes fields containing ';'
summary['commit message'] = summary['commit message'].astype(str).str.replace(r'[\r\n]+', ' ', regex=True).str.strip()
summary.to_csv(f'{netid}_project_summary.csv', sep=';', index=False)

chk = pd.read_csv(f'{netid}_project_summary.csv', sep=';')
print(chk.shape, 'projects:', chk['project_wocid'].nunique())
chk.head()

(17903, 5) projects: 10


,project_wocid,commit_sha1,author,time,commit message
0,antonyuryev_elsevierapi,6f5e25067b51e0571a7fceb59c67f49b37cbcbfe,Anton Yuryev <66886638+AntonYuryev@users.norep...,1612075170,Create README.md
1,antonyuryev_elsevierapi,4f76824db8b2c5eeaed8521b348dfcf95a17626d,Anton Yuryev <66886638+AntonYuryev@users.norep...,1612075203,Update README.md
2,antonyuryev_elsevierapi,92a683db19eea9f4d52b505699bb4fe13d33cea8,Anton Yuryev <66886638+AntonYuryev@users.norep...,1612075670,Add files via upload
3,antonyuryev_elsevierapi,b7fa7975f2c25b3127b7d64750b35f1f85ebf4ea,Anton Yuryev <66886638+AntonYuryev@users.norep...,1612075697,Rename DBObjectTypes.txt to ResnetAPI/DBObject...
4,antonyuryev_elsevierapi,64e7dfd5676fdab70db7e3790979cc34a2e3278e,Anton Yuryev <66886638+AntonYuryev@users.norep...,1612075729,Rename DBPropertyTypes.txt to ResnetAPI/DBProp...


## 6. WoC stats: commits, authors, min/max time

In [8]:
woc_stats = summary.groupby('project_wocid').agg(
    ncommits=('commit_sha1', 'nunique'),
    nauthors=('author', 'nunique'),
    min_time=('time', 'min'),
    max_time=('time', 'max')).reset_index()
woc_stats['from'] = pd.to_datetime(woc_stats['min_time'], unit='s').dt.strftime('%Y-%m-%d')
woc_stats['to']   = pd.to_datetime(woc_stats['max_time'], unit='s').dt.strftime('%Y-%m-%d')


both = woc_stats.merge(gh, left_on='project_wocid', right_on='project').drop(columns='project')
both.to_csv('part1_stats.csv', index=False)
print(both.to_markdown(index=False))

| project_wocid                          |   ncommits |   nauthors |   min_time |   max_time | from       | to         |   nstars |   nforks | lastGHCommitDate   |
|:---------------------------------------|-----------:|-----------:|-----------:|-----------:|:-----------|:-----------|---------:|---------:|:-------------------|
| antonyuryev_elsevierapi                |        733 |          3 | 1612075170 | 1761401938 | 2021-01-31 | 2025-10-25 |      nan |      nan |                    |
| bc_retina                              |        385 |         15 | 1404436477 | 1737352527 | 2014-07-04 | 2025-01-20 |       14 |        3 | 2025-01-20         |
| caseywdunn_siphonophore_phylogeny_2017 |        437 |         15 | 1486818735 | 1729000738 | 2017-02-11 | 2024-10-15 |        2 |        1 | 2024-10-15         |
| giaf_hpipm                             |       1599 |         38 | 1494577210 | 1758727035 | 2017-05-12 | 2025-09-24 |      708 |      144 | 2026-09-18         |
| iic-jku_ddsim 

### Write Up:

### WoC stats

| project_wocid                          |   ncommits |   nauthors |   min_time |   max_time | from       | to         |   nstars |   nforks | lastGHCommitDate   |
|:---------------------------------------|-----------:|-----------:|-----------:|-----------:|:-----------|:-----------|---------:|---------:|:-------------------|
| antonyuryev_elsevierapi                |        733 |          3 | 1612075170 | 1761401938 | 2021-01-31 | 2025-10-25 |      nan |      nan |                    |
| bc_retina                              |        385 |         15 | 1404436477 | 1737352527 | 2014-07-04 | 2025-01-20 |       14 |        3 | 2025-01-20         |
| caseywdunn_siphonophore_phylogeny_2017 |        437 |         15 | 1486818735 | 1729000738 | 2017-02-11 | 2024-10-15 |        2 |        1 | 2024-10-15         |
| giaf_hpipm                             |       1599 |         38 | 1494577210 | 1758727035 | 2017-05-12 | 2025-09-24 |      708 |      144 | 2026-09-18         |
| iic-jku_ddsim                          |       3226 |         47 | 1579629930 | 1762132127 | 2020-01-21 | 2025-11-03 |      162 |       41 | 2026-09-28         |
| malbecc_lio                            |       3272 |         62 | 1191007375 | 1757531069 | 2007-09-28 | 2025-09-10 |       29 |       18 | 2024-01-22         |
| neurodatawithoutborders_matnwb         |       3612 |         42 | 1504116424 | 1761668690 | 2017-08-30 | 2025-10-28 |       62 |       35 | 2026-09-27         |
| prjemian_spec2nexus                    |       1375 |         15 | 1392766189 | 1757329741 | 2014-02-18 | 2025-09-08 |        3 |        7 | 2026-07-20         |
| spirit-code_spirit                     |       3001 |         56 | 1467800336 | 1760450890 | 2016-07-06 | 2025-10-14 |      144 |       60 | 2023-03-17         |
| stanfordnlp_glove                      |        263 |         63 | 1441128078 | 1755526942 | 2015-09-01 | 2025-08-18 |     7230 |     1545 | 2025-07-27         |




### GitHub stats

| project                                |   nstars |   nforks | lastGHCommitDate   |
|:---------------------------------------|---------:|---------:|:-------------------|
| prjemian_spec2nexus                    |        3 |        7 | 2026-07-20         |
| caseywdunn_siphonophore_phylogeny_2017 |        2 |        1 | 2024-10-15         |
| bc_retina                              |       14 |        3 | 2025-01-20         |
| iic-jku_ddsim                          |      162 |       41 | 2026-09-28         |
| spirit-code_spirit                     |      144 |       60 | 2023-03-17         |
| stanfordnlp_glove                      |     7230 |     1545 | 2025-07-27         |
| neurodatawithoutborders_matnwb         |       62 |       35 | 2026-09-27         |
| malbecc_lio                            |       29 |       18 | 2024-01-22         |
| giaf_hpipm                             |      708 |      144 | 2026-09-18         |
| antonyuryev_elsevierapi                |      nan |      nan |        (GitHub repo deleted)            |


## Validation Notes
- 42 out of 17945 commits could not be pulled from WoC even after 3 tries. WoC just does not have them so I left them out of the CSV and the stats.
- The GitHub repo for antonyuryev_elsevierapi was deleted so I could not get its stars or forks or last commit date. WoC still had 733 of its commits.
- The WoC data I got goes up to about November 2025. That is newer than the January 2025 date in the assignment. Some projects kept getting commits on GitHub after that so WoC shows fewer commits for them. These are prjemian_spec2nexus and iic-jku_ddsim and neurodatawithoutborders_matnwb and giaf_hpipm.
- For spirit-code_spirit and malbecc_lio and stanfordnlp_glove the newest WoC commit is newer than the last commit date on GitHub. For spirit WoC has a commit from October 2025 but GitHub says the last commit was in March 2023. I think this is because WoC looks at all branches and forks while GitHub only shows the main branch.
- WoC counts each name and email combo as its own author. So if one person used a few different emails they get counted more than once. That means the WoC author counts might be higher than what GitHub shows.